# Softplus extension of the finite ring-attractor analysis

## Goal

Test how replacing the threshold-linear response function with
softplus changes:

1. Existence of bump fixed points
2. Local stability
3. Dependence of marginal tuning on JE and JI

## Current preliminary findings

- ReLU-tuned JE and JI collapse to a homogeneous fixed point state under the
  beta=2 softplus nonlinearity,instead of generating stable bump solution.
- Joint JE&JI parameter scan reveals a finite region containing stable bump
  candidates.
- Solving the fixed-point equations together with lambda_psi = 0
  gives a preliminary joint marginal-stability curve JE*(JI).

## Possible next steps

- Need to repeat for other N and several beta values.
- Need to compare velocity integration, zero-input drift, and noise
  sensitivity. Other possible property discoveries like limit cycles?

In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt
%pip install scipy

from scipy.optimize import root
from scipy.special import expit

import ra_utilities as util #Still ReLu


importlib.reload(util)

In [ ]:
import scipy
import sys
import os

print(scipy.__version__)
print(sys.executable)
print(os.getpid())

In [ ]:
# Test 
print(util.preferred_headings(8))

In [ ]:
def relu(h):
    return np.maximum(h, 0.0)


def relu_prime(h):
    """Derivative of relu."""
    return (np.asarray(h) > 0.0).astype(float)


def softplus(h, beta=1.0):
    """Smooth approximation to ReLU."""
    h = np.asarray(h)
    return np.logaddexp(0.0, beta * h) / beta


def softplus_prime(h, beta=1.0):
    """Derivative of softplus."""
    return expit(beta * np.asarray(h))
#def sigmoid？

# Plot to show the comparison between softplus and ReLU
x = np.linspace(-4, 4, 500)

plt.figure(figsize=(7, 4))
plt.plot(x, relu(x), label="ReLU", lw=2)

for beta in [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]:
    plt.plot(
        x,
        softplus(x, beta=beta),
        label=fr"Softplus $\beta={beta}$"
    )
# beta value 1.0 or 2.0 would provide better approximation
plt.xlabel(r"$h$")
plt.ylabel(r"$\phi(h)$")
plt.legend()
plt.show()

In [ ]:
def extract_reduced_coordinates(h, theta):
    """Extract H0, a, psi from a network input profile h."""
    h = np.asarray(h)
#Fourier computation
    H0 = np.mean(h)
    H1 = np.mean(h * np.exp(-1j * theta))

    a = 2.0 * np.abs(H1)
    psi = (-np.angle(H1)) % (2.0 * np.pi)

    return H0, a, psi

In [ ]:
# Comparison : ReLU results from article:
N = 6
Nact = 6

# Keep 3/4 ratio for now
# Concept of active neurons only applies to ReLU 

theta = util.preferred_headings(N)

cff = 1.0
tau = 0.1
beta=5

psi_preferred = theta[0]
psi_midpoint = 0.5 * (theta[0] + theta[1])

JE_relu = util.optimal_JE(N, Nact)
JI_relu = util.find_JI_for_amplitude(
    N,
    JE_relu,
    cff=cff,
    target_amplitude=0.2,
)

print("ReLU JE* =", JE_relu)
print("ReLU JI  =", JI_relu)

In [ ]:
def rate_and_gain_projections(
    H0, a, psi, theta, phi, dphi
):
    """Compute the R and Q quantities of the reduced system.These are parameters to 
    simplify the matrix equation and fixed point calculation later on"""

    delta = theta - psi
    cos_delta = np.cos(delta)
    sin_delta = np.sin(delta)

    # h_k = H0 + a cos(theta_k - psi)
    h = H0 + a * cos_delta

    # Firing rate and local nonlinear gain
    r = phi(h)
    gain = dphi(h)

    # Rate projections
    R0 = np.mean(r)
    Rc = np.mean(r * cos_delta)
    Rs = np.mean(r * sin_delta)

    # Gain-weighted moments/projections
    Q0 = np.mean(gain)
    Q1 = np.mean(gain * cos_delta)
    Q2 = np.mean(gain * cos_delta**2)
    Qs = np.mean(gain * sin_delta**2)

    return {
        "h": h,
        "r": r,
        "gain": gain,
        "R0": R0,
        "Rc": Rc,
        "Rs": Rs,
        "Q0": Q0,
        "Q1": Q1,
        "Q2": Q2,
        "Qs": Qs,
    }

In [ ]:
def fixed_point_residual(
    x, JI, JE, cff, psi, theta, phi, dphi
):
    H0, a = x

    q = rate_and_gain_projections(
        H0, a, psi, theta, phi, dphi
    )

    F0 = -H0 + JI * q["R0"] + cff
    Fa = -a + JE * q["Rc"]
    
    return np.array([F0, Fa])

In [ ]:
#A test run: Assign softplus under phi definition,assign random H0 and a)
phi = lambda h: softplus(h, beta=beta)
dphi = lambda h: softplus_prime(h, beta=beta)
def fixed_point_new(
    x, JI, cff, psi, theta, phi, dphi
):
    H0, log_a, JE = x
    a = np.exp(log_a)#to keep a>0 for reasonable bump configuration
    
    q = rate_and_gain_projections(
        H0, a, psi, theta, phi, dphi
    )

    F0 = -H0 + JI * q["R0"] + cff
    Fa = -a + JE * q["Rc"]
    FE = -1 + JE * q["Qs"]
    
    return np.array([F0, Fa,FE])


Given (JI,JE,phi)：

0= -H0+ JI* R0+ cff

0= −a+ JE* Rc
	​


In [ ]:
# Calculate eigenvalues of the Jacobian
def reduced_eigenvalues(
    H0, a, psi, JI, JE, theta, phi, dphi, tau=1.0
):
    q = rate_and_gain_projections(
        H0, a, psi, theta, phi, dphi
    )

    Q0 = q["Q0"]
    Q1 = q["Q1"]
    Q2 = q["Q2"]
    Qs = q["Qs"]

    A = np.array([
        [-1.0 + JI * Q0, JI * Q1],
        [JE * Q1, -1.0 + JE * Q2],
    ])

    lambda_pm_dimensionless = np.linalg.eigvals(A)
    lambda_psi_dimensionless = -1.0 + JE * Qs

    return {
        "A": A,
        "lambda_pm": lambda_pm_dimensionless / tau,
        "lambda_psi": lambda_psi_dimensionless / tau,
        "quantities": q,
    }# Note:return a dictionary, use key to extract
# Use calculated eigenvalues from the dictionary(name:stability)
stability = reduced_eigenvalues(
    H0,
    a,
    psi=psi_preferred,
    JI=JI_relu,
    JE=JE_relu,
    theta=theta,
    phi=phi,
    dphi=dphi,
    tau=tau,
)

print("A =")
print(stability["A"])

print("lambda_pm =", stability["lambda_pm"])
print("lambda_psi =", stability["lambda_psi"])

In [ ]:

# Given JI, solve for H0, a, JE
# satisfying F0 = Fa = FE = 0 at preferred orientation

JI_ini = -6.0
JE_guess = 10.0
psi_preferred = 0.0

solution = root(
    fixed_point_new,
    x0=[-0.8, np.log(1.0), JE_guess],
    args=(
        JI_ini, cff,
        psi_preferred, theta, phi, dphi
    )
)

H0_ini = solution.x[0]
a_ini = np.exp(solution.x[1])
JE_ini = solution.x[2]

q_check = rate_and_gain_projections(
    H0_ini, a_ini, psi_preferred,
    theta, phi, dphi
)

print("Success =", solution.success)
print("Residual norm =", np.linalg.norm(solution.fun))

print("R0 =", q_check["R0"])
print("F0 =", -H0_ini + JI_ini * q_check["R0"] + cff)
print("Fa =", -a_ini + JE_ini * q_check["Rc"])
print("Rs =", q_check["Rs"])
print("FE =", -1 + JE_ini * q_check["Qs"])

print("H0* =", H0_ini)
print("a*  =", a_ini)
print("A*  =", H0_ini + a_ini)
print("JE* =", JE_ini)
print("JI* =", JI_ini)

if not solution.success or np.linalg.norm(solution.fun) > 1e-8:
    print("WARNING: Valid marginal fixed point not confirmed.")


In [ ]:

# Scan JE; solve F0 = Fa = FE = 0
# No fixed A constraint

JE_values = np.linspace(15.6, 20.3, 16)

def find_target_JI(JE_test, previous=None):

    def equations(x):
        H0, log_a = x

        if not np.all(np.isfinite(x)) or not (-20 < log_a < 5):
            return np.array([1e4, 1e4])

        a = np.exp(log_a)

        q = rate_and_gain_projections(
            H0, a, psi_preferred,
            theta, phi, dphi
        )

        Fa_scaled = -1 + JE_test * q["Rc"] / a
        FE = -1 + JE_test * q["Qs"]

        return np.array([Fa_scaled, FE])

    guesses = []

    if previous is not None:
        guesses.append(previous)

    # Broader starting guesses
    for H0_guess in [-0.8, -1.5, -3.0, -5.0, -6.0]:
        for a_guess in [1.0, 2.0, 4.0, 8.0, 16.0]:
            guesses.append([H0_guess, np.log(a_guess)])

    for x0 in guesses:

        solution = root(
            equations,
            x0=x0,
            options={"maxfev": 1000}
        )

        if not solution.success:
            continue

        H0_star, log_a_star = solution.x

        if not (np.log(0.3) < log_a_star < np.log(50)):
            continue

        a_star = np.exp(log_a_star)

        q = rate_and_gain_projections(
            H0_star, a_star, psi_preferred,
            theta, phi, dphi
        )

        # F0 = 0 determines JI
        JI_star = (H0_star - cff) / q["R0"]

        # Verify all three original equations
        residuals = np.array([
            -H0_star + JI_star * q["R0"] + cff,
            -a_star + JE_test * q["Rc"],
            -1 + JE_test * q["Qs"]
        ])

        contrast = phi(H0_star + a_star) - phi(H0_star - a_star)

        if (
            np.all(np.isfinite(residuals))
            and np.linalg.norm(residuals) < 1e-8
            and JI_star < 0
            #and contrast > 0.1
        ):
            return JI_star, a_star, H0_star

    return np.nan, np.nan, np.nan


JI_values = []
a_values = []
H0_values = []

previous = None

for JE in JE_values:

    JI_star, a_star, H0_star = find_target_JI(
        JE, previous=previous
    )

    JI_values.append(JI_star)
    a_values.append(a_star)
    H0_values.append(H0_star)

    if np.all(np.isfinite([JI_star, a_star, H0_star])):
        previous = [H0_star, np.log(a_star)]

JI_values = np.array(JI_values)
a_values = np.array(a_values)
H0_values = np.array(H0_values)

print("JE values:", JE_values)
print("JI values:", JI_values)
print("a values:", a_values)
print("H0 values:", H0_values)
print("A values:", H0_values + a_values)

print("Valid solutions:", np.isfinite(JI_values).sum())


In [ ]:
import pandas as pd
results = []

for i in range(len(JE_values)):

    JE = JE_values[i]
    JI = JI_values[i]
    a = a_values[i]
    H0 = H0_values[i]

    if not np.all(np.isfinite([JE, JI, a, H0])):
        continue

    q = rate_and_gain_projections(
        H0, a, psi_preferred, theta, phi, dphi
    )

    F0 = -H0 + JI*q["R0"] + cff
    Fa = -a + JE*q["Rc"]
    FE = -1 + JE*q["Qs"]

    results.append({
        "JE": JE,
        "JI": JI,
        "H0": H0,
        "a": a,
        "A": H0 + a,
        "F0": F0,
        "Fa": Fa,
        "Rs": q["Rs"],
        "FE": FE
    })

df = pd.DataFrame(results)
display(df)

In [ ]:

# Midpoint Fixed Point Validation
# Keep JE, JI fixed; solve H0, a again at psi = pi/N

results_mid = []

for i in range(len(JE_values)):

    JE = JE_values[i]
    JI = JI_values[i]
    H0_pref = H0_values[i]
    a_pref = a_values[i]

    if not np.all(np.isfinite([JE, JI, H0_pref, a_pref])):
        continue

    # Solve midpoint fixed-point equations: F0 = Fa = 0
    def midpoint_equations(x):

        H0, log_a = x
        a = np.exp(log_a)

        q = rate_and_gain_projections(
            H0, a, psi_midpoint, theta, phi, dphi
        )

        F0 = -H0 + JI * q["R0"] + cff
        Fa = -a + JE * q["Rc"]

        return np.array([F0, Fa])

    solution = root(
        midpoint_equations,
        x0=[H0_pref, np.log(a_pref)]
    )

    H0_mid = solution.x[0]
    a_mid = np.exp(solution.x[1])

    q = rate_and_gain_projections(
        H0_mid, a_mid, psi_midpoint,
        theta, phi, dphi
    )

    F0 = -H0_mid + JI*q["R0"] + cff
    Fa = -a_mid + JE*q["Rc"]
    Rs = q["Rs"]
    FE = -1 + JE*q["Qs"]

    residual = np.linalg.norm([F0, Fa, JE*Rs])

    valid = (
        solution.success
        and np.all(np.isfinite([H0_mid, a_mid, residual]))
        and residual < 1e-8
        and a_mid > 0.1
    )

    results_mid.append({
        "JE": JE,
        "JI": JI,
        "H0_mid": H0_mid if valid else np.nan,
        "a_mid": a_mid if valid else np.nan,
        "A_mid": H0_mid + a_mid if valid else np.nan,
        "F0": F0 if valid else np.nan,
        "Fa": Fa if valid else np.nan,
        "Rs": Rs if valid else np.nan,
        "FE": FE if valid else np.nan,
        "valid": valid
    })

df_mid = pd.DataFrame(results_mid)
display(df_mid.round(6))

#Notice the FE value for midpoint stability


In [ ]:
# 1. Dynamical Equation check:
# for preferred orientation
N = len(theta)

full_residuals = []

for i in range(len(JI_values)):

    JI_star = JI_values[i]
    JE_star = JE_values[i]
    H0_star = H0_values[i]
    a_star = a_values[i]

    if not np.all(np.isfinite([
        JI_star, JE_star, H0_star, a_star
    ])):
        full_residuals.append(np.nan)
        continue

    #Reconstruct neuron input activity:
    h_star = (
        H0_star
        + a_star * np.cos(theta - psi_preferred)
    )

    dhdt = util.network_rhs(
        t=0.0,
        h=h_star,
        N=N,
        JE=JE_star,
        JI=JI_star,
        cff=cff,
        vin=0.0,
        tau=tau,
        phi=phi
    )

    # Calculate full fixed-point residual
    residual_vector = tau * dhdt

    full_residual = np.max(np.abs(residual_vector))
    full_residuals.append(full_residual)

    print(
        f"i={i:2d} | "
        f"JI={JI_star:.4f} | "
        f"JE={JE_star:.4f} | "
        f"a={a_star:.6f} | "
        f"H0={H0_star:.6f} | "
        f"Full residual={full_residual:.3e}"
    )

full_residuals = np.array(full_residuals)

In [ ]:

# 2. Local Stability Analysis: Dynamical Jacobian
import numpy as np

N = len(theta)

lambda_plus_values = []
lambda_minus_values = []
lambda_psi_values = []

J_reduced_values = []
J_full_values = []

for i in range(len(JI_values)):

    JI = JI_values[i]
    JE = JE_values[i]
    H0 = H0_values[i]
    a = a_values[i]

    # Skip invalid solutions
    if not np.all(np.isfinite([JI, JE, H0, a])):
        lambda_plus_values.append(np.nan)
        lambda_minus_values.append(np.nan)
        lambda_psi_values.append(np.nan)
        J_reduced_values.append(None)
        J_full_values.append(None)
        continue

    if full_residuals[i] > 1e-6:
        lambda_plus_values.append(np.nan)
        lambda_minus_values.append(np.nan)
        lambda_psi_values.append(np.nan)
        J_reduced_values.append(None)
        J_full_values.append(None)
        continue

    q = rate_and_gain_projections(
        H0, a, psi_preferred,
        theta, phi, dphi
    )

    Q0 = q["Q0"]
    Q1 = q["Q1"]
    Q2 = q["Q2"]
    Qs = q["Qs"]
    Rc = q["Rc"]

    h_star = q["h"]
    gain = q["gain"]

    W = JI + JE * np.cos(
        theta[:, None] - theta[None, :]
    )
# Compute directly from dynamical equation:
    J_full = (
        -np.eye(N) + W * gain[None, :] / N
    ) / tau

    eig_full = np.linalg.eigvals(J_full)

    J_full_values.append(J_full)

#Reduced dynamical Jacobian (3 x 3) at fixed point:
    if abs(a) < 1e-6:
        lambda_plus_values.append(np.nan)
        lambda_minus_values.append(np.nan)
        lambda_psi_values.append(np.nan)
        J_reduced_values.append(None)

        print(f"i={i}: near homogeneous; use full Jacobian")
        continue

    J_reduced = np.array([
        [-1 + JI*Q0,  JI*Q1,       0],
        [JE*Q1,      -1 + JE*Q2,  0],
        [0,          0,           JE*Qs - JE*Rc/a]
    ]) / tau

    J_reduced_values.append(J_reduced)

#Eigenvalues (H0, a) First two rows
    A = J_reduced[:2, :2]

    eig_trans = np.linalg.eigvals(A)

    eig_trans = eig_trans[
        np.argsort(eig_trans.real)[::-1]
    ]

    lambda_plus = eig_trans[0]
    lambda_minus = eig_trans[1]

    #Phase eigenvalue: lambda_psi：
    lambda_psi = J_reduced[2, 2]

    # Equivalent at an exact stationary bump：#Use the fixed point condition Fa=0
    lambda_psi_FE = (-1 + JE*Qs) / tau


    lambda_plus_values.append(lambda_plus)
    lambda_minus_values.append(lambda_minus)
    lambda_psi_values.append(lambda_psi)

    print(f"\ni = {i}, JI = {JI:.4f}, JE = {JE:.4f}")
    print(f"a = {a:.6f}")

    print(f"lambda_plus  = {lambda_plus:.8g}")
    print(f"lambda_minus = {lambda_minus:.8g}")
    print(f"lambda_psi   = {lambda_psi:.8g}")

    print(f"lambda_psi (FE) = {lambda_psi_FE:.8g}")
    print(f"Full residual   = {full_residuals[i]:.3e}")


lambda_plus_values = np.array(lambda_plus_values)
lambda_minus_values = np.array(lambda_minus_values)
lambda_psi_values = np.array(lambda_psi_values)

In [ ]:

# 2‘. Midpoint Local Stability Analysis: Dynamical Jacobian

import numpy as np

N = len(theta)

lambda_plus_mid_values = []
lambda_minus_mid_values = []
lambda_psi_mid_values = []

J_reduced_mid_values = []
J_full_mid_values = []

full_residuals_mid = []

for i, row in enumerate(df_mid.itertuples(index=False)):

    JE = row.JE
    JI = row.JI
    H0 = row.H0_mid
    a = row.a_mid

    # Skip invalid midpoint solutions
    if (
        not row.valid
        or not np.all(np.isfinite([JE, JI, H0, a]))
        or a < 1e-6
    ):
        lambda_plus_mid_values.append(np.nan)
        lambda_minus_mid_values.append(np.nan)
        lambda_psi_mid_values.append(np.nan)
        J_reduced_mid_values.append(None)
        J_full_mid_values.append(None)
        full_residuals_mid.append(np.nan)
        continue

    q = rate_and_gain_projections(
        H0, a, psi_midpoint,
        theta, phi, dphi
    )

    Q0 = q["Q0"]
    Q1 = q["Q1"]
    Q2 = q["Q2"]
    Qs = q["Qs"]
    Rc = q["Rc"]

    h_star = q["h"]
    gain = q["gain"]

    W = JI + JE * np.cos(
        theta[:, None] - theta[None, :]
    )

    # 1. Validate fixed point in the full network equation
    full_rhs = (
        -h_star + cff + W @ q["r"] / N
    )

    full_residual = np.linalg.norm(full_rhs)
    full_residuals_mid.append(full_residual)

    if full_residual > 1e-6:
        lambda_plus_mid_values.append(np.nan)
        lambda_minus_mid_values.append(np.nan)
        lambda_psi_mid_values.append(np.nan)
        J_reduced_mid_values.append(None)
        J_full_mid_values.append(None)
        continue

    # 2. Full N x N dynamical Jacobian
    J_full = (
        -np.eye(N) + W * gain[None, :] / N
    ) / tau

    eig_full = np.linalg.eigvals(J_full)
    J_full_mid_values.append(J_full)

    # 3. Reduced 3 x 3 Jacobian at midpoint
    J_reduced = np.array([
        [-1 + JI*Q0,  JI*Q1,       0],
        [JE*Q1,      -1 + JE*Q2,  0],
        [0,          0,           JE*Qs - JE*Rc/a]
    ]) / tau

    J_reduced_mid_values.append(J_reduced)

    # Amplitude eigenvalues
    A = J_reduced[:2, :2]

    eig_trans = np.linalg.eigvals(A)
    eig_trans = eig_trans[
        np.argsort(eig_trans.real)[::-1]
    ]

    lambda_plus = eig_trans[0]
    lambda_minus = eig_trans[1]

    # Phase eigenvalue
    lambda_psi = J_reduced[2, 2]

    # Equivalent at stationary bump (Fa = 0)
    lambda_psi_FE = (-1 + JE*Qs) / tau

    lambda_plus_mid_values.append(lambda_plus)
    lambda_minus_mid_values.append(lambda_minus)
    lambda_psi_mid_values.append(lambda_psi)

    print(f"\ni = {i}, JI = {JI:.4f}, JE = {JE:.4f}")
    print(f"H0_mid = {H0:.6f}, a_mid = {a:.6f}")

    print(f"lambda_plus  = {lambda_plus:.8g}")
    print(f"lambda_minus = {lambda_minus:.8g}")
    print(f"lambda_psi   = {lambda_psi:.8g}")

    print(f"lambda_psi (FE) = {lambda_psi_FE:.8g}")
    print(f"Full residual   = {full_residual:.3e}")

# Save midpoint results
lambda_plus_mid_values = np.array(lambda_plus_mid_values)
lambda_minus_mid_values = np.array(lambda_minus_mid_values)
lambda_psi_mid_values = np.array(lambda_psi_mid_values)
full_residuals_mid = np.array(full_residuals_mid)

# Summary table aligned with df_mid
df_mid_stability = df_mid.copy()

df_mid_stability["lambda_plus"] = lambda_plus_mid_values
df_mid_stability["lambda_minus"] = lambda_minus_mid_values
df_mid_stability["lambda_psi"] = lambda_psi_mid_values
df_mid_stability["full_residual"] = full_residuals_mid

df_mid_stability["amplitude_stable"] = (
    (lambda_plus_mid_values.real < 0)
    & (lambda_minus_mid_values.real < 0)
)

df_mid_stability["phase_stable"] = (
    lambda_psi_mid_values < -1e-8
)

display(df_mid_stability.round(6))


In [ ]:

# 4. Phase Drift Analysis
# Preferred-marginality branch: vin = 0

psi0_list = np.linspace(0, np.pi/N, 10)
psi0_deg_list = np.rad2deg(psi0_list)

def angular_error(psi_final, psi_initial):
    return np.abs(
        np.angle(np.exp(1j * (psi_final - psi_initial)))
    )

T_MAX = 50.0
DT = 0.02

drift_results = []

valid_indices = [
    i for i in range(len(JE_values))
    if np.all(np.isfinite([
        JE_values[i], JI_values[i],
        H0_values[i], a_values[i]
    ]))
    and np.real(lambda_plus_values[i]) < 0
    and np.real(lambda_minus_values[i]) < 0
]

ncols = 3
nrows = int(np.ceil(len(valid_indices) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(15, 3 * nrows),
    squeeze=False
)

axes = axes.flatten()

for k, i in enumerate(valid_indices):

    JE = JE_values[i]
    JI = JI_values[i]
    H0 = H0_values[i]
    a = a_values[i]

    ax = axes[k]

    for psi0_deg, psi0 in zip(psi0_deg_list, psi0_list):

        # Rotate the solved preferred bump
        # to a non-preferred initial orientation
        h0 = H0 + a * np.cos(theta - psi0)

        t, h = util.simulate_bump(
            N=N,
            JE=JE,
            JI=JI,
            cff=cff,
            vin=0.0,
            tau=tau,
            t_max=T_MAX,
            dt=DT,
            h0=h0,
            phi=phi,
            return_trajectory=True,
            animate=False,
            method="BDF"
        )

        if (
            len(t) < 2
            or t[-1] < T_MAX - 1e-6
            or not np.all(np.isfinite(h))
        ):
            print(f"Simulation incomplete: JE={JE:.3f}, "
                  f"psi0={psi0_deg} deg")
            continue

        # Extract the first Fourier mode of input activity
        z = (2.0 / N) * (
            h @ np.exp(1j * theta)
        )

        a_trace = np.abs(z)
        psi_trace = np.unwrap(np.angle(z))

        # Phase is undefined if the bump disappears
        if np.min(a_trace) < 1e-3:
            print(f"Bump amplitude too small: JE={JE:.3f}")
            continue

        psi_deg = np.rad2deg(psi_trace)

        ax.plot(
            t, psi_deg,
            label=f"{psi0_deg}°"
        )

        # Final orientation
        psi_final = psi_trace[-1]

        # Fold orientation into the fundamental sector [0, pi/N]
        period = 2 * np.pi / N
        psi_mod = psi_final % period
        psi_fold = min(psi_mod, period - psi_mod)
        phase_error_deg = np.rad2deg(angular_error(psi_final, psi0))
                                                   
        drift_results.append({
            "JE": JE,
            "JI": JI,
            "psi0_deg": psi0_deg,
            "psi_final_deg": np.degrees(psi_final),
            "psi_final_fold_deg": np.degrees(psi_fold),
            "drift_deg": np.degrees(
                psi_final - psi_trace[0]
            ),
            "phase_error_deg": phase_error_deg,
            "a_final": a_trace[-1]
        })

    ax.axhline(
        0, color="gray",
        linestyle="--", alpha=0.4
    )

    ax.axhline(
        180 / N, color="red",
        linestyle="--", alpha=0.4
    )

    ax.set_title(f"JE={JE:.2f}, JI={JI:.2f}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Bump orientation (deg)")
    ax.grid(alpha=0.2)

    if k == 0:
        ax.legend(title="Initial psi", fontsize=8)

# Hide empty panels
for ax in axes[len(valid_indices):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

df_drift = pd.DataFrame(drift_results)
display(df_drift.round(4))


In [ ]:
from pathlib import Path
import pandas as pd


fig_dir = Path("figs") / "Figure2"
fig_dir.mkdir(parents=True, exist_ok=True)

# current model name
model_name = f"softplus_beta{beta:5}"

#model_name = "sigmoid"

fig.suptitle(
    f"Figure 2 | Phase Drift along the Preferred-Marginality Branch\n"
    f"{model_name}, N={N}, T={T_MAX:g} s",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])


fig.savefig(
    fig_dir / f"Figure2_{model_name}_phase_drift.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    fig_dir / f"Figure2_{model_name}_phase_drift.pdf",
    bbox_inches="tight"
)

#plt.show()

print("Saved figure to:", (fig_dir / f"Figure2_{model_name}_phase_drift.png").resolve())
print("Saved figure to:", (fig_dir / f"Figure2_{model_name}_phase_drift.pdf").resolve())


df_drift = pd.DataFrame(drift_results)
display(df_drift.round(4))

df_drift.to_csv(
    fig_dir / f"Figure2_{model_name}_phase_drift.csv",
    index=False,
    float_format="%.16g"
)

print("Saved drift table to:", (fig_dir / f"Figure2_{model_name}_phase_drift.csv").resolve())

In [ ]:

# Step 1. Solve stationary bump at fixed JE and JI
# Only solve F0 = Fa = 0
# Do NOT impose FE = 0 or fixed amplitude

def solve_bump_fixed_connectivity(
    JE,
    JI,
    seed_H0,
    seed_a,
    psi=psi_preferred,
    residual_tol=1e-7,
    a_min=0.1,
    contrast_min=0.1,
    stability_tol=1e-8
):

    # Use log(a) to ensure positive amplitude
    def equations_logamp(x):

        H0, log_a = x

        # Numerical protection
        if (
            not np.all(np.isfinite(x))
            or not (-15 < log_a < 15)
        ):
            return np.array([1e6, 1e6])

        a = np.exp(log_a)

        q = rate_and_gain_projections(
            H0, a, psi,
            theta, phi, dphi
        )

        F0 = -H0 + JI*q["R0"] + cff

        # Scaled amplitude equation
        # Fa/a = -1 + JE*Rc/a
        Fa_scaled = -1 + JE*q["Rc"]/a #Necessary modification for a to not be homogeneous 0 solution

        return np.array([F0, Fa_scaled])

    # Standardized invalid return
    invalid_result = {
        "JE": JE,
        "JI": JI,
        "H0": np.nan,
        "a": np.nan,
        "A": np.nan,
        "F0": np.nan,
        "Fa": np.nan,
        "fp_residual": np.inf,
        "full_residual": np.inf,
        "rate_contrast": np.nan,
        "lambda_plus": np.nan,
        "lambda_minus": np.nan,
        "lambda_psi": np.nan,
        "transverse_stable": False,
        "phase_stable": False,
        "full_stable": False,
        "solver_success": False,
        "valid": False
    }

    if not np.all(np.isfinite([seed_H0, seed_a])) or seed_a <= 0:
        return invalid_result.copy()


    
    # Multiple initial guesses
    # Start near the reference bump branch

    guesses = [
        [seed_H0, np.log(seed_a)],
        [seed_H0, np.log(0.7*seed_a)],
        [seed_H0, np.log(1.5*seed_a)],
        [seed_H0, np.log(2.0*seed_a)]
    ]

    solution = None

    for x0 in guesses:

        trial = root(
            equations_logamp,
            x0=x0,
            method="hybr",
            options={"maxfev": 2000, "xtol": 1e-10}
        )

        H0_try, log_a_try = trial.x

        # Reject invalid numerical results
        if (
            not np.all(np.isfinite(trial.x))
            or not (-15 < log_a_try < 15)
        ):
            continue

        a_try = np.exp(log_a_try)

        # Exclude near-homogeneous solutions
        if a_try <= a_min:
            continue

        # Check scaled equations
        scaled_residual = np.max(
            np.abs(equations_logamp(trial.x))
        )

        if (
            not np.isfinite(scaled_residual)
            or scaled_residual > 1e-8
        ):
            continue

        # Verify the ORIGINAL fixed-point equations
        F0_try, Fa_try = fixed_point_residual(
            [H0_try, a_try],
            JI, JE, cff,
            psi, theta, phi, dphi
        )

        original_residual = np.linalg.norm(
            [F0_try, Fa_try]
        )

        if (
            not np.isfinite(original_residual)
            or original_residual > residual_tol
        ):
            continue

        # Accept a nonzero stationary bump candidate
        solution = trial
        break


    # No acceptable nonzero root found
    if solution is None:
        return invalid_result.copy()
    
    H0, log_a = solution.x
    a = np.exp(log_a)

    q = rate_and_gain_projections(
        H0, a, psi,
        theta, phi, dphi
    )

    # Original fixed-point equations
    F0, Fa = fixed_point_residual(
        [H0, a],
        JI, JE, cff,
        psi, theta, phi, dphi
    )

    # Original fixed-point residual
    fp_residual = np.linalg.norm([F0, Fa])

    # Verify against the full N-neuron dynamical equation
    # Use the existing util.network_rhs
    

    h_star = H0 + a * np.cos(theta - psi)

    dhdt = util.network_rhs(
        t=0.0,
        h=h_star,
        N=N,
        JE=JE,
        JI=JI,
        cff=cff,
        vin=0.0,
        tau=tau,
        phi=phi
    )

    full_residual = np.max(
        np.abs(tau * dhdt)
    )

    # Actual firing-rate contrast across neurons
    rate_contrast = np.ptp(q["r"])# np.ptp: peak-to-peak function, check whether the bump has identifiable profile


    # Reduced Jacobian eigenvalues
    # Reuse the existing reduced_eigenvalues function
    stability = reduced_eigenvalues(
        H0=H0,
        a=a,
        psi=psi,
        JI=JI,
        JE=JE,
        theta=theta,
        phi=phi,
        dphi=dphi,
        tau=tau
    )

    eig_pm = np.asarray(stability["lambda_pm"])

    eig_pm = eig_pm[
        np.argsort(eig_pm.real)[::-1]
    ]

    lambda_plus = eig_pm[0].real
    lambda_minus = eig_pm[1].real

    lambda_psi = np.real(
        stability["lambda_psi"]
    )

    # Full N x N Jacobian
    # Follow the construction in your original cell 32

    W = JI + JE * np.cos(
        theta[:, None] - theta[None, :]
    )

    J_full = (
        -np.eye(N) + W * q["gain"][None, :] / N
    ) / tau

    eig_full = np.linalg.eigvals(J_full)

    transverse_stable = (
        lambda_plus < -stability_tol
        and lambda_minus < -stability_tol
    )

    phase_stable = (
        lambda_psi < -stability_tol
    )

    full_stable = np.all(
        eig_full.real < -stability_tol
    )

    # Final validity check
    valid = (
        np.all(np.isfinite([
            H0, a, fp_residual,
            full_residual, rate_contrast
        ]))
        and fp_residual < residual_tol
        and full_residual < residual_tol
        and a > a_min
        and rate_contrast > contrast_min
    )

    return {
        "JE": JE,
        "JI": JI,
        "H0": H0,
        "a": a,
        "A": H0 + a,

        "F0": F0,
        "Fa": Fa,

        "fp_residual": fp_residual,
        "full_residual": full_residual,
        "rate_contrast": rate_contrast,

        "lambda_plus": lambda_plus,
        "lambda_minus": lambda_minus,
        "lambda_psi": lambda_psi,

        "transverse_stable": transverse_stable,
        "phase_stable": phase_stable,
        "full_stable": full_stable,

        "solver_success": solution.success,
        "valid": valid
    }


In [ ]:

# Step 2. Two-dimensional connectivity perturbation
# Fix JE and JI independently, then solve F0 = Fa = 0

i_star = 3

JE_star = JE_values[i_star]
JI_star = JI_values[i_star]
H0_star = H0_values[i_star]
a_star = a_values[i_star]

print("Reference parameters:")
print(f"JE* = {JE_star:.6f}")
print(f"JI* = {JI_star:.6f}")
print(f"H0* = {H0_star:.6f}")
print(f"a*  = {a_star:.6f}")

# Step 2.1: Test the reference parameter pair
reference = solve_bump_fixed_connectivity(
    JE=JE_star,
    JI=JI_star,
    seed_H0=H0_star,
    seed_a=a_star
)

if not reference["valid"]:
    raise RuntimeError(
        "Reference bump is invalid. "
        "Check the solver before scanning."
    )

# Step 2.2: Relative connectivity perturbations
eta_E_values = np.linspace(-0.05, 0.05, 5)
eta_I_values = np.linspace(-0.05, 0.05, 5)

scan_results = []

for eta_I in eta_I_values:

    for eta_E in eta_E_values:

        # Perturb JE and JI independently
        JE = JE_star * (1 + eta_E)

        JI = JI_star + abs(JI_star) * eta_I

        # Re-solve H0 and a without FE or fixed A
        row = solve_bump_fixed_connectivity(
            JE=JE,
            JI=JI,
            seed_H0=H0_star,
            seed_a=a_star
        )

        # Ensure every result contains these fields
        default_fields = {
            "lambda_plus": np.nan,
            "lambda_minus": np.nan,
            "lambda_psi": np.nan,
            "transverse_stable": False,
            "phase_stable": False,
            "full_stable": False
        }

        row = {**default_fields, **row}

        # Save relative perturbations
        row["eta_E"] = eta_E
        row["eta_I"] = eta_I

        scan_results.append(row)

# Step 2.3: Collect the results
df_scan = pd.DataFrame(scan_results)

print("\nTotal parameter pairs:", len(df_scan))
print("Valid stationary bumps:", df_scan["valid"].sum())
print("Transversely stable:",
      df_scan["transverse_stable"].sum())

# Step 2.4: Display
columns_to_show = [
    "eta_E", "eta_I",
    "JE", "JI",
    "H0", "a", "A",
    "fp_residual",
    "full_residual",
    "rate_contrast",
    "lambda_plus",
    "lambda_minus",
    "lambda_psi",
    "transverse_stable",
    "phase_stable",
    "full_stable",
    "valid"
]

display(df_scan[columns_to_show].round(6))


In [ ]:
# Keep consistent criteria for all reference points
A_MIN = 0.05
CONTRAST_MIN = 0.05

reference_results = []

for i in range(len(JE_values)):

    JE_ref = JE_values[i]
    JI_ref = JI_values[i]
    H0_ref = H0_values[i]
    a_ref = a_values[i]

    # Skip failed solutions from the original scan
    if not np.all(np.isfinite([
        JE_ref, JI_ref, H0_ref, a_ref
    ])):
        continue

    if a_ref <= 0:
        continue

    # Validate the reference with the existing solver
    ref = solve_bump_fixed_connectivity(
        JE=JE_ref,
        JI=JI_ref,
        seed_H0=H0_ref,
        seed_a=a_ref,
        psi=psi_preferred,
        a_min=A_MIN,
        contrast_min=CONTRAST_MIN
    )

    # Require an accepted nonzero stationary bump
    if not ref["valid"]:
        continue

    # Require amplitude/mean-input stability
    if not ref["transverse_stable"]:
        continue

    # Check how close the reference is to marginality
    q = rate_and_gain_projections(
        ref["H0"], ref["a"],
        psi_preferred,
        theta, phi, dphi
    )

    FE_ref = -1 + JE_ref*q["Qs"]

    reference_results.append({
        "i_ref": i,
        "JE_ref": JE_ref,
        "JI_ref": JI_ref,
        "H0_ref": ref["H0"],
        "a_ref": ref["a"],
        "A_ref": ref["A"],
        "FE_ref": FE_ref,
        "lambda_psi_ref": ref["lambda_psi"],
        "rate_contrast_ref": ref["rate_contrast"]
    })

df_refs = pd.DataFrame(reference_results)

if df_refs.empty:
    raise RuntimeError(
        "No valid reference bump found. "
        "Check solver and reference arrays."
    )

print("Number of valid reference candidates:", len(df_refs))

display(df_refs.round(6))


In [ ]:

# STEP 2. 2D connectivity perturbation
# around ALL reference candidates

# Start with a small grid for each reference
eta_E_values = np.linspace(-0.05, 0.05, 5)
eta_I_values = np.linspace(-0.05, 0.05, 5)

scan_results = []

for _, ref in df_refs.iterrows():

    i_ref = int(ref["i_ref"])

    JE_ref = ref["JE_ref"]
    JI_ref = ref["JI_ref"]
    H0_ref = ref["H0_ref"]
    a_ref = ref["a_ref"]

    print(
        f"Scanning reference {i_ref}: "
        f"JE={JE_ref:.4f}, JI={JI_ref:.4f}"
    )

    # Already solved bumps near this reference
    known_bumps = []

    # Start at the center, then move outward
    grid_points = [
        (float(eta_E), float(eta_I))
        for eta_I in eta_I_values
        for eta_E in eta_E_values
    ]

    grid_points.sort(
        key=lambda p: p[0]**2 + p[1]**2
    )

    for eta_E, eta_I in grid_points:

        # Independently perturb the two couplings
        JE = JE_ref * (1 + eta_E)

        JI = JI_ref + abs(JI_ref)*eta_I

        # Initial seed from the reference
        seeds = [(H0_ref, a_ref)]

        # Additional seeds from nearby solved bumps
        nearby = sorted(
            known_bumps,
            key=lambda r:
                (r["eta_E"] - eta_E)**2
                + (r["eta_I"] - eta_I)**2
        )[:3]

        seeds += [
            (r["H0"], r["a"])
            for r in nearby
        ]

        candidates = []

        for H0_seed, a_seed in seeds:

            result = solve_bump_fixed_connectivity(
                JE=JE,
                JI=JI,
                seed_H0=H0_seed,
                seed_a=a_seed,
                psi=psi_preferred,
                a_min=A_MIN,
                contrast_min=CONTRAST_MIN
            )

            if result["valid"]:
                candidates.append(result)

        # Prefer transversely stable candidates
        stable_candidates = [
            r for r in candidates
            if r["transverse_stable"]
        ]

        pool = (
            stable_candidates
            if stable_candidates
            else candidates
        )

        if pool:

            # Select a solution close to the branch
            # already tracked near this parameter point
            target = (
                nearby[0]
                if nearby
                else {"H0": H0_ref, "a": a_ref}
            )

            row = min(
                pool,
                key=lambda r:
                    ((r["H0"] - target["H0"])
                     / (1 + abs(target["H0"])))**2
                    + np.log(
                        r["a"] / target["a"]
                    )**2
            ).copy()

            if row["transverse_stable"]:
                stationary_status = "stable_bump"

                known_bumps.append({
                    **row,
                    "eta_E": eta_E,
                    "eta_I": eta_I
                })
            else:
                stationary_status = "transverse_unstable"

        else:

            # No acceptable root found by current search
            row = {
                "JE": JE,
                "JI": JI,
                "H0": np.nan,
                "a": np.nan,
                "A": np.nan,
                "rate_contrast": np.nan,
                "fp_residual": np.nan,
                "full_residual": np.nan,
                "lambda_plus": np.nan,
                "lambda_minus": np.nan,
                "lambda_psi": np.nan,
                "valid": False,
                "transverse_stable": False,
                "phase_stable": False,
                "full_stable": False
            }

            stationary_status = "no_accepted_bump"

        # Attach the reference identity
        row["i_ref"] = i_ref
        row["JE_ref"] = JE_ref
        row["JI_ref"] = JI_ref

        # Relative coordinates of this perturbation
        row["eta_E"] = eta_E
        row["eta_I"] = eta_I

        row["stationary_status"] = stationary_status

        scan_results.append(row)

# Combine the perturbations of ALL reference centers
df_scan = pd.DataFrame(scan_results)

# Give every test a unique identity
df_scan["scan_i"] = np.arange(len(df_scan))

print("\nTotal reference centers:", len(df_refs))
print("Total tested parameter points:", len(df_scan))

print("\nStationary-bump status:")
print(df_scan["stationary_status"].value_counts())

display(
    df_scan[[
        "i_ref", "scan_i",
        "eta_E", "eta_I",
        "JE", "JI",
        "H0", "a", "A",
        "rate_contrast",
        "lambda_plus",
        "lambda_minus",
        "lambda_psi",
        "stationary_status"
    ]].round(6)
)


In [ ]:

# STEP 2'. Midpoint fixed-point and stability analysis
# For every perturbed JE, JI pair in df_scan
from scipy.optimize import root

psi_midpoint = psi_preferred + np.pi/N
STABILITY_TOL = 1e-8

def solve_midpoint_fixed_point(row):

    JE = row["JE"]
    JI = row["JI"]

    H0_pref = row["H0"]
    a_pref = row["a"]

    # Default result if no acceptable midpoint bump is found
    output = {
        "scan_i": int(row["scan_i"]),
        "H0_mid": np.nan,
        "a_mid": np.nan,
        "F0_mid": np.nan,
        "Fa_mid": np.nan,
        "Rs_mid": np.nan,
        "full_residual_mid": np.nan,
        "lambda_plus_mid": np.nan,
        "lambda_minus_mid": np.nan,
        "lambda_psi_mid": np.nan,
        "amplitude_stable_mid": False,
        "phase_stable_mid": False,
        "mid_valid": False
    }

    if (
        not bool(row["valid"])
        or not np.all(np.isfinite([
            JE, JI, H0_pref, a_pref
        ]))
        or a_pref <= 0
    ):
        return output

    # Solve F0 = 0 and Fa/a = 0 at midpoint
    def midpoint_equations(x):

        H0, log_a = x

        if (
            not np.all(np.isfinite(x))
            or not (-15 < log_a < 15)
        ):
            return np.array([1e6, 1e6])

        a = np.exp(log_a)

        q = rate_and_gain_projections(
            H0, a, psi_midpoint,
            theta, phi, dphi
        )

        F0 = -H0 + JI*q["R0"] + cff
        Fa_scaled = -1 + JE*q["Rc"]/a

        return np.array([F0, Fa_scaled])

    # Multiple starting guesses
    guesses = [
        [H0_pref, np.log(a_pref)],
        [H0_pref, np.log(0.7*a_pref)],
        [H0_pref, np.log(1.5*a_pref)],
        [H0_pref - 0.2, np.log(a_pref)],
        [H0_pref + 0.2, np.log(a_pref)]
    ]

    candidates = []

    W = JI + JE*np.cos(
        theta[:, None] - theta[None, :]
    )

    for x0 in guesses:

        sol = root(
            midpoint_equations,
            x0=x0,
            method="hybr",
            options={"maxfev": 2000, "xtol": 1e-10}
        )

        if not np.all(np.isfinite(sol.x)):
            continue

        H0_mid, log_a_mid = sol.x

        if not (-15 < log_a_mid < 15):
            continue

        a_mid = np.exp(log_a_mid)

        if a_mid <= A_MIN:
            continue

        q = rate_and_gain_projections(
            H0_mid, a_mid, psi_midpoint,
            theta, phi, dphi
        )

        F0 = -H0_mid + JI*q["R0"] + cff
        Fa = -a_mid + JE*q["Rc"]
        Rs = q["Rs"]

        # Full N-neuron stationary equation
        full_rhs_vector = (
            -q["h"] + cff + W @ q["r"]/N
        )

        full_residual = np.max(
            np.abs(full_rhs_vector)
        )

        contrast = np.ptp(q["r"])

        scaled_residual = np.max(
            np.abs(midpoint_equations(sol.x))
        )

        if not (
            np.isfinite(full_residual)
            and full_residual < 1e-7
            and scaled_residual < 1e-8
            and contrast > CONTRAST_MIN
        ):
            continue

        # Reduced Jacobian at reflection-symmetric midpoint
        Q0 = q["Q0"]
        Q1 = q["Q1"]
        Q2 = q["Q2"]
        Qs = q["Qs"]
        Rc = q["Rc"]

        J_reduced = np.array([
            [-1 + JI*Q0, JI*Q1, 0],
            [JE*Q1, -1 + JE*Q2, 0],
            [0, 0, JE*Qs - JE*Rc/a_mid]
        ]) / tau

        # Mean/amplitude eigenvalues
        eig_trans = np.linalg.eigvals(
            J_reduced[:2, :2]
        )

        eig_trans = eig_trans[
            np.argsort(eig_trans.real)[::-1]
        ]

        lambda_plus = eig_trans[0].real
        lambda_minus = eig_trans[1].real

        # Phase eigenvalue
        lambda_psi = J_reduced[2, 2]

        # Full network Jacobian
        J_full = (
            -np.eye(N)
            + W*q["gain"][None, :]/N
        ) / tau

        eig_full = np.linalg.eigvals(J_full)

        candidates.append({
            "H0_mid": H0_mid,
            "a_mid": a_mid,
            "F0_mid": F0,
            "Fa_mid": Fa,
            "Rs_mid": Rs,
            "full_residual_mid": full_residual,
            "lambda_plus_mid": lambda_plus,
            "lambda_minus_mid": lambda_minus,
            "lambda_psi_mid": lambda_psi,
            "amplitude_stable_mid": bool(
                lambda_plus < -STABILITY_TOL
                and lambda_minus < -STABILITY_TOL
            ),
            "phase_stable_mid": bool(
                lambda_psi < -STABILITY_TOL
            ),
            "full_stable_mid": bool(
                np.all(eig_full.real < -STABILITY_TOL)
            )
        })

    if not candidates:
        return output

    # Choose the root closest to the preferred bump branch
    best = min(
        candidates,
        key=lambda r:
            (r["H0_mid"] - H0_pref)**2
            + (r["a_mid"] - a_pref)**2
    )

    output.update(best)
    output["mid_valid"] = True

    return output


# Apply to ALL perturbation pairs
mid_results = []

for _, row in df_scan.iterrows():

    mid_result = solve_midpoint_fixed_point(row)
    mid_results.append(mid_result)

df_mid_scan = pd.DataFrame(mid_results)

# Combine preferred and midpoint stability
df_compare = df_scan.merge(
    df_mid_scan,
    on="scan_i",
    how="left"
)

display(
    df_compare[[
        "i_ref", "scan_i",
        "JE", "JI",
        "lambda_plus", "lambda_minus", "lambda_psi",
        "H0_mid", "a_mid",
        "lambda_plus_mid",
        "lambda_minus_mid",
        "lambda_psi_mid",
        "mid_valid"
    ]].round(6)
)


In [ ]:

# STEP 3A. Phase drift after connectivity perturbations
# Simulate and store trajectories only

psi0_deg_list = [5, 10, 15, 20, 25]
psi0_list = np.deg2rad(psi0_deg_list)

T_MAX = 50.0
DT = 0.02

# Accepted, transversely stable stationary bumps
valid_networks = df_compare[
    df_compare["valid"]
    & df_compare["transverse_stable"]
].copy()

valid_networks = valid_networks.sort_values(
    ["i_ref", "eta_I", "eta_E"]
)

print("Networks to simulate:", len(valid_networks))

phase_trajectories = []

for _, row in valid_networks.iterrows():

    JE = row["JE"]
    JI = row["JI"]
    H0 = row["H0"]
    a = row["a"]

    for psi0_deg, psi0 in zip(
        psi0_deg_list, psi0_list
    ):

        # Rotate the preferred stationary bump
        h0 = H0 + a*np.cos(theta - psi0)

        try:
            t, h = util.simulate_bump(
                N=N,
                JE=JE,
                JI=JI,
                cff=cff,
                vin=0.0,
                tau=tau,
                t_max=T_MAX,
                dt=DT,
                h0=h0,
                phi=phi,
                return_trajectory=True,
                animate=False,
                method="BDF"
            )

        except Exception as exc:
            print(
                f"Simulation failed: "
                f"JE={JE:.3f}, JI={JI:.3f}, "
                f"psi0={psi0_deg}°: {exc}"
            )
            continue

        t = np.asarray(t)
        h = np.asarray(h)

        if (
            len(t) < 2
            or t[-1] < T_MAX - 1e-6
            or h.shape != (len(t), N)
            or not np.all(np.isfinite(h))
        ):
            print(
                f"Incomplete simulation: "
                f"JE={JE:.3f}, JI={JI:.3f}"
            )
            continue

        # Fourier extraction of input activity
        z = (2.0/N) * (
            h @ np.exp(1j*theta)
        )

        a_trace = np.abs(z)
        psi_trace = np.unwrap(np.angle(z))

        # Verify that the bump survives
        contrast_trace = np.ptp(phi(h), axis=1)

        if (
            np.min(a_trace) < A_MIN
            or np.min(contrast_trace) < CONTRAST_MIN
        ):
            continue

        phase_trajectories.append({
            "scan_i": int(row["scan_i"]),
            "i_ref": int(row["i_ref"]),
            "JE": JE,
            "JI": JI,
            "psi0_deg": psi0_deg,
            "t": t,
            "psi_deg": np.rad2deg(psi_trace),
            "a_trace": a_trace
        })

print("Stored trajectories:", len(phase_trajectories))


# STEP 3B. Plot phase trajectories grouped by reference
# One figure per reference, with shared legend and caption

from collections import defaultdict
from matplotlib.lines import Line2D

PLOT_MODE = "orientation"
# "orientation": psi(t)
# "drift": psi(t) - psi0

# Consistent colors for initial orientations
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
psi_colors = {
    deg: colors[k]
    for k, deg in enumerate(psi0_deg_list)
}

# Group stored trajectories by network
traces_by_scan = defaultdict(list)

for tr in phase_trajectories:
    traces_by_scan[int(tr["scan_i"])].append(tr)

reference_indices = sorted(
    df_compare["i_ref"].unique()
)

for i_ref in reference_indices:

    subset = df_compare[
        df_compare["i_ref"] == i_ref
    ].copy()

    # Original reference connectivity
    JE_ref = subset["JE_ref"].iloc[0]
    JI_ref = subset["JI_ref"].iloc[0]

    # 2D perturbation grid
    eta_E_grid = np.sort(subset["eta_E"].unique())
    eta_I_grid = np.sort(subset["eta_I"].unique())

    ncols = len(eta_E_grid)
    nrows = len(eta_I_grid)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(3.6*ncols, 2.8*nrows),
        squeeze=False,
        sharex=True,
        sharey=(PLOT_MODE == "orientation")
    )


    for row_idx, eta_I in enumerate(eta_I_grid):

        for col_idx, eta_E in enumerate(eta_E_grid):

            ax = axes[row_idx, col_idx]

            selected = subset[
                np.isclose(subset["eta_E"], eta_E)
                & np.isclose(subset["eta_I"], eta_I)
            ]

            if selected.empty:
                ax.axis("off")
                continue

            row = selected.iloc[0]
            scan_i = int(row["scan_i"])

            trajectories = sorted(
                traces_by_scan.get(scan_i, []),
                key=lambda tr: tr["psi0_deg"]
            )

            for tr in trajectories:

                t = tr["t"]
                psi_deg = tr["psi_deg"]
                psi0_deg = tr["psi0_deg"]

                if PLOT_MODE == "orientation":
                    y = psi_deg
                else:
                    y = psi_deg - psi0_deg

                ax.plot(
                    t,
                    y,
                    color=psi_colors[psi0_deg],
                    linewidth=1.5
                )

            # Reference orientation lines
            if PLOT_MODE == "orientation":

                ax.axhline(
                    0,
                    color="gray",
                    linestyle="--",
                    linewidth=1,
                    alpha=0.5
                )

                ax.axhline(
                    180/N,
                    color="red",
                    linestyle="--",
                    linewidth=1,
                    alpha=0.5
                )

                ax.set_ylim(-1, 180/N + 2)

            else:

                ax.axhline(
                    0,
                    color="gray",
                    linestyle="--",
                    linewidth=1,
                    alpha=0.5
                )

            # Mark failed or invalid networks
            if len(trajectories) == 0:

                ax.text(
                    0.5, 0.5,
                    "No valid trajectory",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="gray",
                    fontsize=9
                )

            # Center of this perturbation grid
            is_reference = (
                np.isclose(eta_E, 0)
                and np.isclose(eta_I, 0)
            )

            ref_label = " [REF]" if is_reference else ""

            # Local phase eigenvalues
            lp = row["lambda_psi"]
            lm = row["lambda_psi_mid"]

            ax.set_title(
                f"ΔJE={eta_E*100:+.1f}% | "
                f"ΔJI={eta_I*100:+.1f}%"
                f"{ref_label}\n"
                f"λp={lp:.2g}, λm={lm:.2g}",
                fontsize=9
            )

            ax.grid(alpha=0.2)

            if row_idx == nrows - 1:
                ax.set_xlabel("Time (s)")

            if col_idx == 0:
                ax.set_ylabel(
                    "Bump orientation (deg)"
                    if PLOT_MODE == "orientation"
                    else "Phase displacement (deg)"
                )

   
    # Main figure title
    

    fig.suptitle(
        f"Reference {int(i_ref)} | "
        f"JE*={JE_ref:.3f}, JI*={JI_ref:.3f}\n"
        "Phase dynamics under connectivity perturbations",
        fontsize=15,
        fontweight="bold",
        y=0.99
    )

    legend_handles = [
        Line2D(
            [0], [0],
            color=psi_colors[deg],
            linewidth=2,
            label=f"ψ₀ = {deg}°"
        )
        for deg in psi0_deg_list
    ]

    if PLOT_MODE == "orientation":

        legend_handles += [
            Line2D(
                [0], [0],
                color="gray",
                linestyle="--",
                label="Preferred (0°)"
            ),
            Line2D(
                [0], [0],
                color="red",
                linestyle="--",
                label=f"Midpoint ({180/N:.0f}°)"
            )
        ]

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.915),
        ncol=len(legend_handles),
        fontsize=10,
        frameon=False
    )
  
    if PLOT_MODE == "orientation":

        caption_text = (
            f"N={N}, vin=0, T={T_MAX:g} s. "
            "Columns: JE perturbation; rows: JI perturbation. "
            "λp and λm are local phase eigenvalues "
            "at preferred and midpoint fixed points."
        )

    else:

        caption_text = (
            f"N={N}, vin=0, T={T_MAX:g} s. "
            "Phase displacement is measured "
            "relative to each initial orientation."
        )

    fig.text(
        0.5, 0.015,
        caption_text,
        ha="center",
        va="bottom",
        fontsize=10,
        color="dimgray"
    )

    # Reserve space for title, legend and caption
    fig.subplots_adjust(
        top=0.85,
        bottom=0.08,
        left=0.07,
        right=0.98,
        hspace=0.50,
        wspace=0.25
    )

    plt.show()



In [ ]:

# STEP 4A. Summarize drift performance for each network
import numpy as np
import pandas as pd

traj_summary_rows = []

for tr in phase_trajectories:

    psi_deg = np.asarray(tr["psi_deg"])
    t = np.asarray(tr["t"])

    if len(psi_deg) < 2:
        continue

    # Use extracted initial phase, not the nominal psi0,
    # to avoid tiny numerical offsets
    psi_init = psi_deg[0]

    disp_deg = psi_deg - psi_init

    final_disp_deg = disp_deg[-1]
    max_abs_disp_deg = np.max(np.abs(disp_deg))

    traj_summary_rows.append({
        "scan_i": int(tr["scan_i"]),
        "i_ref": int(tr["i_ref"]),
        "JE": tr["JE"],
        "JI": tr["JI"],
        "psi0_deg": tr["psi0_deg"],
        "final_disp_deg": final_disp_deg,
        "abs_final_disp_deg": abs(final_disp_deg),
        "max_abs_disp_deg": max_abs_disp_deg
    })

df_traj_summary = pd.DataFrame(traj_summary_rows)

if df_traj_summary.empty:
    raise RuntimeError("No valid phase trajectories stored in phase_trajectories.")

display(df_traj_summary.round(6))


# STEP 4B. Aggregate to network-level performance

N_PSI_REQUIRED = len(psi0_deg_list)

LOW_DRIFT_THRESH = 0.10
# 

df_perf = (
    df_traj_summary
    .groupby(["scan_i", "i_ref", "JE", "JI"], as_index=False)
    .agg(
        n_traj=("psi0_deg", "count"),
        E_max_final_deg=("abs_final_disp_deg", "max"),
        E_rms_final_deg=("final_disp_deg", lambda x: np.sqrt(np.mean(np.square(x)))),
        E_max_traj_deg=("max_abs_disp_deg", "max")
    )
)

df_perf["all_orientations_measured"] = (
    df_perf["n_traj"] == N_PSI_REQUIRED
)

df_perf["low_drift"] = (
    df_perf["all_orientations_measured"]
    & (df_perf["E_max_final_deg"] <= LOW_DRIFT_THRESH)
)

display(df_perf.round(6))


# STEP 4C. Merge performance back into df_compare


df_map = df_compare.merge(
    df_perf,
    on=["scan_i", "i_ref", "JE", "JI"],
    how="left"
)

df_map["n_traj"] = df_map["n_traj"].fillna(0)
df_map["all_orientations_measured"] = df_map["all_orientations_measured"].fillna(False)
df_map["low_drift"] = df_map["low_drift"].fillna(False)

display(df_map.round(6))

In [ ]:

# STEP 4E. Combined JE-JI performance heatmaps
# All reference candidates in ONE figure

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize

LOW_DRIFT_THRESH = 0.5

# Recompute low-drift criteria
df_map["low_drift"] = (
    df_map["valid"].fillna(False)
    & df_map["transverse_stable"].fillna(False)
    & df_map["all_orientations_measured"].fillna(False)
    & (df_map["E_max_final_deg"] <= LOW_DRIFT_THRESH)
)


reference_indices = sorted(df_map["i_ref"].unique())

ncols = 3
nrows = int(np.ceil(len(reference_indices) / ncols))

valid_errors = df_map["E_max_final_deg"].dropna()

if valid_errors.empty:
    raise RuntimeError("No valid drift measurements found.")

# Common color range across ALL references
vmin = 0
vmax = max(valid_errors.max(), 1e-8)

norm = Normalize(vmin=vmin, vmax=vmax)

cmap = plt.get_cmap("viridis_r").copy()
cmap.set_bad("lightgray")


fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(16, 4.0 * nrows),
    squeeze=False
)

axes = axes.flatten()
for k, i_ref in enumerate(reference_indices):

    ax = axes[k]

    sub = df_map[
        df_map["i_ref"] == i_ref
    ].copy()

    # Full perturbation grid, including missing values
    heat = sub.pivot(
        index="eta_I",
        columns="eta_E",
        values="E_max_final_deg"
    ).sort_index().sort_index(axis=1)

    low = sub.pivot(
        index="eta_I",
        columns="eta_E",
        values="low_drift"
    ).reindex(
        index=heat.index,
        columns=heat.columns
    ).fillna(False)

    Z = np.ma.masked_invalid(
        heat.to_numpy(dtype=float)
    )

    # Heatmap with shared color normalization
    ax.imshow(
        Z,
        origin="lower",
        aspect="auto",
        cmap=cmap,
        norm=norm
    )


    low_matrix = low.to_numpy(dtype=bool)

    for row_idx, col_idx in np.argwhere(low_matrix):

        ax.add_patch(
            Rectangle(
                (col_idx - 0.5, row_idx - 0.5),
                1, 1,
                fill=False,
                edgecolor="red",
                linewidth=2
            )
        )

    ref_row = np.where(
        np.isclose(heat.index.to_numpy(), 0)
    )[0]

    ref_col = np.where(
        np.isclose(heat.columns.to_numpy(), 0)
    )[0]

    if len(ref_row) and len(ref_col):

        ax.scatter(
            ref_col[0],
            ref_row[0],
            marker="+",
            s=140,
            color="black",
            linewidths=2.5,
            zorder=5
        )
    JE_ref = sub["JE_ref"].iloc[0]
    JI_ref = sub["JI_ref"].iloc[0]

    JE_ticks = JE_ref * (
        1 + heat.columns.to_numpy()
    )

    JI_ticks = JI_ref + abs(JI_ref) * (
        heat.index.to_numpy()
    )

    ax.set_xticks(np.arange(len(JE_ticks)))
    ax.set_xticklabels(
        [f"{x:.2f}" for x in JE_ticks],
        rotation=45,
        ha="right",
        fontsize=9
    )

    ax.set_yticks(np.arange(len(JI_ticks)))
    ax.set_yticklabels(
        [f"{y:.2f}" for y in JI_ticks],
        fontsize=9
    )

    ax.set_xlabel("JE", fontsize=10)
    ax.set_ylabel("JI", fontsize=10)

    n_low = int(low_matrix.sum())
    n_total = low_matrix.size

    ax.set_title(
        f"Reference {int(i_ref)} | "
        f"JE*={JE_ref:.2f}, JI*={JI_ref:.2f}\n"
        f"Low-drift points: {n_low}/{n_total}",
        fontsize=11
    )

# Hide extra panels
for ax in axes[len(reference_indices):]:
    ax.axis("off")

fig.subplots_adjust(
    left=0.07,
    right=0.88,
    top=0.92,
    bottom=0.10,
    hspace=0.70,
    wspace=0.35
)

cax = fig.add_axes([0.905, 0.17, 0.018, 0.65])

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=norm
)

sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)

cbar.set_label(
    "Maximum final phase error (deg)",
    fontsize=12
)

legend_handles = [
    Patch(
        facecolor="none",
        edgecolor="red",
        linewidth=2,
        label=f"Low drift (error ≤ {LOW_DRIFT_THRESH}°)"
    ),
    Line2D(
        [0], [0],
        marker="+",
        color="black",
        linestyle="None",
        markersize=12,
        markeredgewidth=2,
        label="Unperturbed reference"
    ),
    Patch(
        facecolor="lightgray",
        label="No valid measurement"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.48, 0.045),
    ncol=3,
    fontsize=11,
    frameon=False
)


fig.suptitle(
    "Softplus | Connectivity Tuning and Phase Drift\n"
    "Low-drift regions across marginality candidates",
    fontsize=17,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.48, 0.025,
    f"N={N}, vin=0, T=50 s. "
    "Each panel shows independent JE and JI perturbations "
    "around one reference candidate.",
    ha="center",
    fontsize=10,
    color="dimgray"
)

plt.show()


In [ ]:
from pathlib import Path

# Create the fig folder if it doesn't exist
fig_dir = Path("figs")
fig_dir.mkdir(parents=True, exist_ok=True)
# Save the entire combined figure
fig.savefig(
    fig_dir / "softplus_beta5_tuning_heatmaps.png",
    dpi=300,
    bbox_inches="tight"
)

# Vector version, suitable for manuscripts
fig.savefig(
    fig_dir / "softplus_beta5_tuning_heatmaps.pdf",
    bbox_inches="tight"
)


In [ ]:
# # # pass the solved JE value back to bump simulation and see the validity of solution
# JE_simu=19.2
# JI_simu=-35

# # h_traj is the FULL trajectory
# t, h_traj = util.simulate_bump(
#     N=N,
#     JE=JE_simu,
#     JI=JI_simu,
#     cff=cff,
#     vin=0.0,
#     tau=tau,
#     t_max=3.0,
#     dt=0.01,
#     h0=-2+2.014226* np.cos(theta - 1.309),
#     phi=phi,
#     return_trajectory=True,
#     animate=True,
#     method="BDF"
# )

# # Extract coordinates at the FINAL time point
# H0_final, a_final, psi_final = extract_reduced_coordinates(
#     h_traj[-1], theta
# )

# print("Final state:")
# print("H0 =", H0_final)
# print("a  =", a_final)
# print("psi (deg) =", np.degrees(psi_final))


In [ ]:

from pathlib import Path
import json


model_name = f"softplus_beta{beta:5}"

data_dir = Path("data") / model_name
data_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "references.csv": df_refs,
    "fixedpoint_stability.csv": df_compare,
    "tuning_performance.csv": df_map
}

for filename, dataframe in datasets.items():
    dataframe.to_csv(
        data_dir / filename,
        index=False,
        float_format="%.16g"
    )

config = {
    "model": model_name,
    "N": int(N),
    "cff": float(cff),
    "tau": float(tau),
    "simulation_T": float(T_MAX),
    "simulation_dt": float(DT),
    "psi0_deg": [float(x) for x in psi0_deg_list],
    "eta_E": eta_E_values.tolist(),
    "eta_I": eta_I_values.tolist(),
    "low_drift_threshold_deg": float(LOW_DRIFT_THRESH)
}

with open(data_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Research data saved to:", data_dir.resolve())
